# AgentRegistry, end to end: scaffold a dice agent, add approved MCP tools, deploy to kagent **and** AWS Bedrock AgentCore, then govern the tools

A developer builds an agent with `arctl`, runs it locally, pulls in **approved MCP tool servers** from the registry catalog, publishes the agent, and deploys the *same* published agent to two runtimes (Solo Enterprise for kagent on kind, and AWS Bedrock AgentCore). Finally a platform owner restricts which MCP tools the agent may call with an **AccessPolicy**, enforced at an agentgateway waypoint.

> **Kernel:** pick **Bash** (top-right). One-time engineer setup lives in `scripts/` (see `README.md`).

## What you're running

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 980 700" width="100%" font-family="-apple-system,Segoe UI,Helvetica,Arial,sans-serif">
  <style>
    .ns{rx:10;ry:10;stroke-width:1.5}
    .pill{fill:#ffffff;stroke:#cbd5e1;stroke-width:1;rx:7;ry:7}
    .t{font-size:13px;fill:#0f172a}
    .nt{font-size:13px;font-weight:700}
    .sub{font-size:11px;fill:#475569}
    .lbl{font-size:11px;fill:#334155}
    .ar{stroke:#64748b;stroke-width:1.6;fill:none;marker-end:url(#a)}
  </style>
  <defs><marker id="a" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#64748b"/></marker></defs>

  <!-- kind cluster -->
  <rect x="20" y="58" width="630" height="620" rx="14" fill="#f8fafc" stroke="#1e293b" stroke-width="2"/>
  <text x="38" y="82" class="nt" fill="#1e293b">kind cluster &#183; agentcore-demo</text>
  <text x="38" y="99" class="sub">ambient mesh + agentgateway ingress &#8212; consoles at http://*.localtest.me</text>

  <!-- agentregistry-system ns (the in-cluster control plane) -->
  <rect x="38" y="112" width="594" height="86" class="ns" fill="#ede9fe" stroke="#6d28d9"/>
  <text x="52" y="132" class="nt" fill="#6d28d9">namespace: agentregistry-system &#8212; AgentRegistry control plane (in-cluster)</text>
  <rect x="52" y="144" width="168" height="42" class="pill"/><text x="62" y="162" class="t">registry server</text><text x="62" y="177" class="sub">UI/API :12121</text>
  <rect x="232" y="144" width="168" height="42" class="pill"/><text x="242" y="162" class="t">Postgres + ClickHouse</text><text x="242" y="177" class="sub">bundled</text>
  <rect x="412" y="144" width="208" height="42" class="pill"/><text x="422" y="162" class="t">catalog + runtimes</text><text x="422" y="177" class="sub">MCP/skills &#183; kagent + AgentCore + GCP</text>

  <!-- kagent ns -->
  <rect x="38" y="212" width="594" height="150" class="ns" fill="#e0f2fe" stroke="#0369a1"/>
  <text x="52" y="232" class="nt" fill="#0369a1">namespace: kagent &#8212; Solo Enterprise for kagent (runtime #1)</text>
  <rect x="52" y="244" width="150" height="42" class="pill"/><text x="62" y="262" class="t">kagent controller</text><text x="62" y="277" class="sub">+ OBO, OIDC</text>
  <rect x="214" y="244" width="130" height="42" class="pill"/><text x="224" y="262" class="t">agentdemo</text><text x="224" y="277" class="sub">ADK agent pod</text>
  <rect x="356" y="244" width="132" height="42" class="pill"/><text x="366" y="262" class="t">agentgateway</text><text x="366" y="277" class="sub">waypoint (enforce)</text>
  <rect x="52" y="298" width="190" height="42" class="pill"/><text x="62" y="316" class="t">MCP: everything-server</text><text x="62" y="331" class="sub">sum/echo/printenv&#8230;</text>
  <rect x="254" y="298" width="150" height="42" class="pill"/><text x="264" y="316" class="t">MCP: my-mcp</text><text x="264" y="331" class="sub">word_count</text>
  <rect x="416" y="298" width="204" height="42" class="pill"/><text x="426" y="316" class="t">AccessPolicy (deny printenv)</text><text x="426" y="331" class="sub">kagent CR, enforced at waypoint</text>

  <!-- solo-enterprise ns -->
  <rect x="38" y="376" width="594" height="104" class="ns" fill="#fef3c7" stroke="#92400e"/>
  <text x="52" y="396" class="nt" fill="#92400e">namespace: solo-enterprise &#8212; management + observability</text>
  <rect x="52" y="408" width="160" height="42" class="pill"/><text x="62" y="426" class="t">Enterprise UI</text><text x="62" y="441" class="sub">kagent.localtest.me</text>
  <rect x="224" y="408" width="160" height="42" class="pill"/><text x="234" y="426" class="t">OTel collector</text><text x="234" y="441" class="sub">OTLP :4317</text>
  <rect x="396" y="408" width="150" height="42" class="pill"/><text x="406" y="426" class="t">ClickHouse</text><text x="406" y="441" class="sub">otel_traces_json</text>
  <text x="52" y="470" class="sub">agent spans &#9656; collector &#9656; ClickHouse &#9656; Enterprise UI Tracing tab</text>

  <!-- keycloak ns -->
  <rect x="38" y="494" width="594" height="62" class="ns" fill="#dcfce7" stroke="#166534"/>
  <text x="52" y="514" class="nt" fill="#166534">namespace: keycloak &#8212; single sign-on</text>
  <text x="52" y="534" class="sub">realm &#8220;agentregistry&#8221; &#183; issuer keycloak.localtest.me &#183; admin-user / password (admins &#8594; Admin)</text>
  <text x="52" y="549" class="sub">one login for arctl, the registry server, the kagent controller and the Enterprise UI</text>

  <!-- ingress / mesh note -->
  <rect x="38" y="570" width="594" height="96" class="ns" fill="#f1f5f9" stroke="#475569"/>
  <text x="52" y="590" class="nt" fill="#334155">agentgateway-system: ingress + ambient mesh + image registry (:5001)</text>
  <text x="52" y="610" class="sub">ingress Gateway routes *.localtest.me &#8594; registry / Enterprise UI / Keycloak (no port-forward)</text>
  <text x="52" y="628" class="sub">ztunnel enrols kagent into the ambient mesh; the waypoint authorizes MCP tool calls vs the AccessPolicy</text>
  <text x="52" y="646" class="sub">Anthropic Claude is the model for the agent on all three runtimes</text>

  <!-- developer arctl (external) -->
  <rect x="690" y="112" width="268" height="72" rx="12" fill="#1e293b"/>
  <text x="706" y="136" class="nt" fill="#f8fafc">arctl (developer CLI)</text>
  <text x="706" y="156" class="sub" fill="#cbd5e1">init / build / run (local) &#183; apply</text>
  <text x="706" y="174" class="sub" fill="#cbd5e1">user login &#8594; in-cluster registry</text>

  <!-- Anthropic -->
  <rect x="690" y="206" width="268" height="64" rx="12" fill="#f5f5f4" stroke="#78350f"/>
  <text x="706" y="230" class="nt" fill="#78350f">Anthropic Claude</text>
  <text x="706" y="250" class="sub">model &#183; claude-haiku-4-5</text>

  <!-- AWS AgentCore -->
  <rect x="690" y="292" width="268" height="118" rx="12" fill="#fff7ed" stroke="#9a3412"/>
  <text x="706" y="316" class="nt" fill="#9a3412">AWS Bedrock AgentCore</text>
  <text x="706" y="336" class="sub">runtime #2 (external)</text>
  <text x="706" y="354" class="sub">clones agent source from git,</text>
  <text x="706" y="370" class="sub">runs the same published agent</text>
  <text x="706" y="392" class="sub">connected once at setup (CF role + ECR)</text>

  <!-- Google Cloud / Vertex AI -->
  <rect x="690" y="432" width="268" height="118" rx="12" fill="#e8f0fe" stroke="#1a73e8"/>
  <text x="706" y="456" class="nt" fill="#1a73e8">Google Cloud (Vertex AI)</text>
  <text x="706" y="476" class="sub">runtime #3 (external)</text>
  <text x="706" y="494" class="sub">agent &#8594; Vertex AI Agent Engine,</text>
  <text x="706" y="510" class="sub">MCP tool &#8594; Cloud Run</text>
  <text x="706" y="532" class="sub">built from git; MCP linked via deploymentRefs</text>

  <!-- arrows -->
  <path class="ar" d="M690,150 C668,150 660,152 634,152"/><text x="684" y="142" class="lbl" text-anchor="end">publish + deploy</text>
  <path class="ar" d="M650,238 L690,238"/><text x="654" y="232" class="lbl">model</text>
  <path class="ar" d="M650,330 C672,330 678,345 690,345"/><text x="648" y="360" class="lbl" text-anchor="end">deploy #2</text>
  <path class="ar" d="M650,470 C672,470 678,490 690,490"/><text x="648" y="500" class="lbl" text-anchor="end">deploy #3</text>
</svg>

> The developer drives everything through **arctl** &#8594; the **in-cluster AgentRegistry** control plane (`agentregistry-system`), which deploys the *same* published agent to three runtimes &#8212; **kagent** in this cluster, **AWS Bedrock AgentCore**, and **Google Cloud** (Vertex AI). One **Keycloak** login spans the platform; the whole UI is reached at **http://*.localtest.me** through the agentgateway ingress (no port-forwards); agent **traces** flow to **ClickHouse** and render in the **Enterprise UI**; MCP tool calls are governed by an **AccessPolicy** at an **agentgateway waypoint**.

## Open the consoles

Run this once **in a terminal** to open the consoles, served by the agentgateway ingress at `*.localtest.me` (no port-forwards):

```bash
./scripts/open-consoles.sh
```

| Console | URL | Login | What it's for |
|---|---|---|---|
| **AgentRegistry UI** | **http://agentregistry.localtest.me** | **admin-user / password** | catalog, runtimes, deployed instances |
| **Enterprise UI** | **http://kagent.localtest.me** | **admin-user / password** | Dashboard, Agents, Tracing, Access Policies |
| Keycloak (admin) | http://keycloak.localtest.me | admin / admin | manage the IdP (realm, clients, users) |
| AWS Bedrock AgentCore | the AWS console (manual) | — | the second runtime |
| Google Cloud (Vertex AI) | the Google Cloud console (manual) | — | the third runtime |

> **Login is `admin-user` / `password`** for both UIs — a Keycloak realm user in the `admins` group (registry superuser + kagent Admin). The `arctl` CLI logs in as the same user automatically (the Connect cell runs `arctl user login`). The Keycloak **admin** console itself is `admin / admin`.
> Tracing lives in the Enterprise UI; chat with the agent from the CLI (`./scripts/ask.sh "…"`) or in the Enterprise UI.

## Connect to the platform

Loads creds, puts `arctl` on the path, mints a catalog token.

In [53]:
rm -rf agentdemo

In [54]:
[ -d agentregistry-agentcore-kind ] && cd agentregistry-agentcore-kind || :; source scripts/connect.sh

arctl v2026.5.4 · alice token ok · cluster kind-agentcore-demo
NAME            TYPE
aws-agentcore   BedrockAgentCore
kind-kagent     Kagent
local           Local


## 1. Browse the registry: approved tools, skills, and runtimes

Rather than wiring arbitrary MCP servers, developers pull from an **approved catalog**. Open the **AgentRegistry UI** (http://agentregistry.localtest.me) → **Tool Servers**, or list it from the CLI. While we're here, list the **skills** (reusable instruction packs) and the **runtimes** the registry can deploy to — `kind-kagent` (Solo Enterprise for kagent) and `aws-agentcore` (AWS Bedrock AgentCore), both connected at setup.

In [55]:
arctl get mcpservers
echo
arctl get skills
echo
arctl get runtimes

NAME                     TAG      DESCRIPTION
acme/textkit             latest   Text utilities for summarization agents — word_count an...
demo/everything-server   latest   Approved org tool server: sum, echo, uppercase, reverse.
demo/my-mcp              latest   Text utilities tool server: word_count.

NAME                  TAG      DESCRIPTION
dice-game             latest   How to run a fair dice game and explain prime results cle...
release-notes-style   latest   The house format for release notes. One headline, grouped...
summary-style         latest   The house format for summaries — length budget, bullets...

NAME            TYPE
aws-agentcore   BedrockAgentCore
kind-kagent     Kagent
local           Local


## 2. Create the agent, wired to ONE approved tool

Scaffold the agent and reference **one** approved server from the catalog with `--mcp` — `my-mcp` (the `word_count` tool). We'll **add a second one later** to show a tool coming online. arctl records the reference in `agent.yaml` under `spec.mcpServers`. The generated agent also ships two **local** tools, `roll_die` and `check_prime`.

In [56]:
arctl init agent agentdemo --framework adk --language python --model-provider anthropic --model-name claude-haiku-4-5 --mcp my-mcp@latest

✓ Created agent: agentdemo (framework: adk, language: python, model: anthropic/claude-haiku-4-5)

🚀 Next steps:
  1. Run locally (optional):
     arctl run agentdemo
     (export ANTHROPIC_API_KEY in your shell or set it in .env first)
  2. Publish to the registry:
     arctl apply -f agentdemo/agent.yaml


In [ ]:
cat agentdemo/agentdemo/agent.py

In [ ]:
cat agentdemo/agent.yaml

## 3. Build and run it locally

Build the image:

In [57]:
arctl build ./agentdemo

→ adk-python: docker build -t localhost:5001/agentdemo:latest /Users/thomasorourke/code/solo/solo-demos/agentregistry-agentcore-kind/agentdemo
[+] Building 0.0s (0/1)                                          docker:default
[+] Building 0.2s (1/2)                                          docker:default
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 429B                                       0.0s
 => [internal] load metadata for ghcr.io/kagent-dev/kagent/kagent-adk:0.8  0.2s
[+] Building 0.3s (1/3)                                          docker:default
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 429B                                       0.0s
 => [internal] load metadata for ghcr.io/kagent-dev/kagent/kagent-adk:0.8  0.3s
 => [auth] kagent-dev/kagent/kagent-adk:pull token for ghcr.io             0.0s
[+] Building 0.4s (2/3)                                  

Then chat with the agent **in a terminal** — `arctl run` is interactive, so copy the command below and run it in your shell:

```bash
arctl run ./agentdemo
```

Ask it (paste into the chat):

```text
Roll a 20-sided die and tell me whether the result is prime.
```

It uses the local `roll_die` + `check_prime` tools. The approved catalog tool `word_count` (from `my-mcp`) comes online when the agent is **deployed** — the registry stands the MCP server up beside it (you'll see it next). We add `everything-server` (`sum`, `echo`, …) later.

## 4. Build (multi-arch), publish, and push the source

Multi-arch so the same image runs on kagent (arm64 here) and AgentCore (amd64).

In [58]:
arctl build ./agentdemo --platform linux/amd64,linux/arm64 --push

→ adk-python: docker build --platform=linux/amd64,linux/arm64 -t localhost:5001/agentdemo:latest /Users/thomasorourke/code/solo/solo-demos/agentregistry-agentcore-kind/agentdemo
[+] Building 0.0s (0/1)                                          docker:default
[+] Building 0.2s (2/4)                                          docker:default
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 429B                                       0.0s
 => [linux/arm64 internal] load metadata for ghcr.io/kagent-dev/kagent/ka  0.2s
 => [linux/amd64 internal] load metadata for ghcr.io/kagent-dev/kagent/ka  0.2s
 => [auth] kagent-dev/kagent/kagent-adk:pull token for ghcr.io             0.0s
[+] Building 0.4s (2/4)                                          docker:default
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 429B                                       0.0s
 => [linux/arm64 inter

In [59]:
arctl apply -f agentdemo/agent.yaml

→ Injecting labels from arctl.yaml: arctl.dev/framework=adk, arctl.dev/language=python
✓ Agent/agentdemo (latest) created


AgentCore clones the agent **source** from git at deploy time, so push it to your agent repo (the engineer set `AGENT_GIT_URL` in `.env.local`):

In [60]:
./scripts/git-push.sh 2>&1   # 2>&1 so the script's messages show in the notebook (this kernel hides stderr)

✓ pushed agentdemo/ source -> tjorourke/agentregistry-agentcore-demo@main


## 5. Kick off the AgentCore deploy (runtime #2) — in the background

The same agent, now to a second runtime: **AWS Bedrock AgentCore** (connected once at setup). The cell below signs in to AWS and gives the in-cluster registry your AWS creds in the **foreground** (~30-45s), then runs the slow part — build, push to ECR, deploy — in the **background** so you can carry straight on with §6 while AgentCore provisions (~2-4 min).

AgentCore publishes its **own** agent record (`agentdemo-agentcore`, ECR image), so it never collides with the kagent `agentdemo` agent. Watch it any time with `tail -f /tmp/agentcore-deploy.log`; **§8** invokes it and waits for READY.

In [ ]:
source scripts/agentcore.sh

## 5b. ...and to Google Cloud (runtime #3) — also in the background

The *same* published agent, now to a **third** runtime: **Google Cloud** — the agent on
**Vertex AI Agent Engine**, its MCP tool on **Cloud Run**. Connected once at setup with
`./scripts/04e-connect-gcp.sh` (a `GeminiAgentRuntime`).

Google is not quite like the other two. Unlike kagent, the agent's MCP tools are **not**
auto-wired: the MCP is deployed to Cloud Run first and linked from the agent's Deployment
via `deploymentRefs`. So the whole sequence (MCP → wait → agent) is backgrounded together.
The cell returns immediately; Vertex provisions (~8-12 min) while you carry on. §8b waits
for it to be READY.

In [ ]:
source scripts/gcp.sh &

## 6. Deploy onto kagent and work with it (runtime #1)

Now the in-cluster runtime. Deploy the agent and its approved MCP tool server onto **kagent**, then explore it end to end: list its tools, add another approved tool and watch the list grow, then run a real task and watch the trace in the Enterprise UI.

In [ ]:
# Deploy the my-mcp tool server onto kind-kagent (its own Deployment).
arctl apply -f yaml/deploy-mcp-my-mcp.yaml
echo "waiting for my-mcp to be Ready..."
kubectl --context kind-agentcore-demo -n kagent wait --for=condition=Ready mcpserver/my-mcp --timeout=180s
kubectl --context kind-agentcore-demo -n kagent get mcpserver

In [ ]:
# Deploy the agent onto kagent. The registry binds it to kind-kagent and derives
# the agent's MCP_SERVERS_CONFIG from the MCP servers deployed on the runtime
# (that's why we deployed my-mcp first).
# The model key is NEVER passed in cleartext: the registry Deployment env is a
# literal string map (no secretKeyRef), so this script deploys the agent with no
# key, then patches the kagent Agent CR to read ANTHROPIC_API_KEY from the
# kagent-anthropic Secret via env.valueFrom.secretKeyRef.
# (No image juggling needed: AgentCore in §5 publishes its own agent record, so
# this agentdemo agent stays on the localhost image.)
./scripts/deploy-agent-kagent.sh

In [ ]:
kubectl --context kind-agentcore-demo -n kagent get pods

### Ask what tools the agent has

Right now the agent has `word_count` (from `my-mcp`) plus its two local tools — we add `everything-server` next. Ask on the CLI:

In [ ]:
./scripts/ask.sh "List the exact names of every tool you can call. Output only a comma-separated list, nothing else."

### Add another approved tool — watch it appear

`agentdemo` was scaffolded with one tool (`my-mcp`). The agent's tools are **declared** in `agent.yaml`, so you add another approved server by declaring it and re-applying — in the **UI** (Catalog → `agentdemo` → **Edit** → add `everything-server`), or by pasting the snippet in the next cell into `agentdemo/agent.yaml` under `spec.mcpServers:`.

Then run the cell after it to apply, deploy the MCP server, and re-deploy the agent — all `arctl`, no rebuild. (The kagent MCP server is named `everything-server` from the catalog ref either way, so §8's policy targets it identically.)

In [ ]:
# After adding everything-server to the agent (UI Edit, or paste the cell above into agent.yaml):
arctl apply -f agentdemo/agent.yaml                      # re-publish the agent (now with both tools)
arctl apply -f yaml/deploy-mcp-everything-server.yaml    # deploy the everything-server MCP to kagent
kubectl --context kind-agentcore-demo -n kagent wait --for=condition=Ready mcpserver/everything-server --timeout=180s

# re-deploy the agent (re-derives its tool list) — key still sourced from the Secret, never cleartext:
./scripts/deploy-agent-kagent.sh

### Ask again — the new tools appear

Same question, and now `sum`, `echo`, `printenv`, `reverse_text`, `to_uppercase` are in the list (alongside `word_count` + the local tools). The tool came online by adding it to the spec and redeploying — no rebuild:

In [ ]:
./scripts/ask.sh "List the exact names of every tool you can call. Output only a comma-separated list, nothing else."

### Now run a real task and watch the trace

Listing tools is one thing — now watch the agent actually **call** them. Give `agentdemo` a multi-step task on the CLI (or chat in the Enterprise UI). It exercises the whole chain (a local tool, an MCP tool, then another local tool):

In [ ]:
./scripts/ask.sh "Roll a 13-sided die, add 5 to the result, then tell me if it is prime."

Now open the **Enterprise UI** (http://kagent.localtest.me) → **Tracing**. The run shows up as a span tree — `invocation → call_llm → generate_content (claude-haiku) → execute_tool roll_die → sum → check_prime` — with the model and token usage on each LLM span. (The full OTEL trace lives in the Enterprise UI.)

## 7. Govern the MCP tools with an AccessPolicy (in the UI)

The `everything-server` exposes a sensitive `printEnv` tool. A platform owner restricts the agent to least privilege — **in the UI**.

First, see what the agent can call today — ask on the CLI:

```bash
./scripts/ask.sh "What tools do you have?"
```

You'll see the full set, including `printEnv`.

Create the policy in the **Solo Enterprise UI** (http://kagent.localtest.me) → **Access Policies** → **Create New Access Policy**. Do the steps **in order** — the **Select Agent** picker only populates once you've set the cluster + namespace in step 1:

1. **Policy (1/4):** give it a name; **Cluster = `agentcore-demo`**; **Namespace = `kagent`**.
2. **Subjects (2/4):** Subject Kind **Agent** → **Select Agent** → **`agentdemo`**.
3. **Action (3/4):** **ALLOW**; tools: **`sum`**.
4. **Target (4/4):** Target Type **MCP Server** → **`everything-server`**.

This is an allowlist — on the everything-server the agent may now call **only** `sum`; `printEnv`, `echo` and the rest are denied.

> This lab is a single co-located cluster, so the Enterprise UI manages it as the **local cluster** (`agentcore-demo`) directly — no relay, and it appears once. (Prefer the CLI? `./scripts/accesspolicy-on.sh` applies the identical policy. The AgentRegistry UI at agentregistry.localtest.me can also create it.)

The MCP server is **already** behind an agentgateway waypoint (the registry labels it `kagent.solo.io/waypoint=true` at deploy time) — that waypoint is the enforcement point. This makes it explicit / idempotent:

In [ ]:
# The everything-server is already waypoint-labelled by the registry; confirm/ensure it.
kubectl --context kind-agentcore-demo -n kagent label mcpserver everything-server kagent.solo.io/waypoint=true --overwrite
kubectl --context kind-agentcore-demo -n kagent rollout status deploy/mcpserver-everything-server-waypoint --timeout=120s 2>/dev/null || true
echo "✓ everything-server is behind the agentgateway waypoint — the AccessPolicy is enforced there"

In [ ]:
# The policy you created in the UI is a declarative kagent CR — show it:
kubectl --context kind-agentcore-demo -n kagent get accesspolicy
echo
kubectl --context kind-agentcore-demo -n kagent get accesspolicy -o yaml

Ask again on the CLI — the everything-server now offers only `sum`; `printEnv` and the rest are gone:

> A new chat re-lists the tools through the waypoint — no restart needed. (If the list looks unchanged, the agent is still on its startup tool cache: `kubectl --context kind-agentcore-demo -n kagent rollout restart deploy/agentdemo`.)

In [ ]:
./scripts/ask.sh "What tools do you have?"

## 8. Invoke the same agent on AgentCore (runtime #2)

You kicked off the AgentCore deploy back in §5; by now the runtime has provisioned. Invoke it with the same kind of task — this runs the *identical* governed agent in **AWS**, not your cluster. `ac-invoke.sh` waits for the runtime to be **READY** first, so if it's still finishing it pauses a moment, then prints the answer.

In [ ]:
./scripts/ac-invoke.sh "Roll a 13-sided die, add 5 to the result, then tell me if it is prime."

## 8b. Invoke the same agent on Google (runtime #3)

You kicked off the Google deploy in §5b; by now Vertex AI has provisioned. Invoke it with
the same kind of prompt — it waits for the runtime to be READY, then queries the Vertex AI
Agent Engine. Same agent, same tools, third runtime.

In [ ]:
./scripts/gcp-invoke.sh "Roll a 13-sided die, add 5 to the result, then tell me if it is prime."

## Reset / teardown

Run any of these **in a terminal** (copy each line):

```bash
./scripts/accesspolicy-off.sh   # restore the full tool list
./scripts/reset.sh              # back to start (clears agentdemo, deployments)
./scripts/cleanup.sh            # full teardown (cluster, registry, AWS)
```